# 09 — סימולציית מונטה קרלו לקואליציות

**השאלה:** מה הסיכוי שקואליציה מסוימת תגיע ל-61 מנדטים?

ממוצע הסקרים נותן מספר אחד לכל מפלגה, אבל לסקרים יש אי-ודאות. אם הליכוד עומד על 21.7 עם פיזור של כמה מנדטים, ומפלגה אחרת עומדת על 3.3, ממש על סף אחוז החסימה, אז "גוש X מקבל 60" יכול בפועל להיות 57 או 63. **סימולציית מונטה קרלו** מגרילה אלפי "בחירות" אפשריות שמתאימות לאי-הוודאות הזו, וסופרת באיזה חלק מהן הקואליציה עוברת את 61.

**חשוב — מה המחברת לא עושה:**
- **היא לא מחשבת את הממוצע בעצמה.** הממוצע מגיע מ-`web/scripts/poll-stats.ts`, אותו קוד שהאתר מריץ. כך אין במחברת עותק בפייתון של הלוגיקה.
- **היא מקרבת את חלוקת המנדטים.** מפלגה מתחת לאחוז החסימה מקבלת 0, והשאר מנורמלות ל-120. האתר משתמש בבדר-עופר המלא מ-`lib/seats.ts`, והמחברת היא המקום לבחור את **מודל אי-הוודאות**.

התא הראשון מכין את הנתונים: הממוצע לכל מפלגה, ומטריצה של הסקרים מ-14 הימים האחרונים. בה כל שורה היא סקר, וכל עמודה היא מפלגה. מפלגה שלא הופיעה בסקר נספרת כ-0, כמו בממוצע.

In [1]:
import json, subprocess, sqlite3
import numpy as np
import pandas as pd

stats = json.loads(subprocess.run(
    ["node", "scripts/poll-stats.ts", "--window", "14", "--json"],
    cwd="../web", capture_output=True, text=True, check=True).stdout)
parties = pd.DataFrame(stats["parties"]).set_index("id")
avg = parties["average"]
as_of = pd.Timestamp(stats["asOf"])

con = sqlite3.connect("../data/election.db")
rows = pd.read_sql('''SELECT p.id AS poll, p.fieldDateEnd / 1000 AS t, r.partyId AS party, r.seats
                      FROM Poll p JOIN PollResult r ON r.pollId = p.id''', con)
rows = rows[pd.to_datetime(rows["t"], unit="s") > as_of - pd.Timedelta(days=14)]
polls = rows.pivot_table(index="poll", columns="party", values="seats", fill_value=0).reindex(columns=avg.index, fill_value=0)

TOTAL, MAJORITY = 120, 61
THRESHOLD_SEATS = TOTAL * 0.0325      # 3.9 מנדטים
print(f"ממוצע נכון ל-{stats['asOf']}, {len(polls)} סקרים ב-14 הימים האחרונים")
pd.DataFrame({"average": avg, "bloc": parties["bloc"]}).sort_values("average", ascending=False)

ממוצע נכון ל-2026-10-15, 16 סקרים ב-14 הימים האחרונים


,average,bloc
id,,
likud,21.7,right
yashar,21.7,center
yachad,11.1,center
democrats,9.0,center-left
joint-list,8.1,arab
yahadut-hatorah,8.0,haredi
otzma,7.9,right
yisrael-beitenu,7.8,right
shas,7.7,haredi


## משימה 1 — כמה כל מפלגה "זזה"?

לכל מפלגה, חשב את **סטיית התקן** של המרחק בין כל סקר לבין הממוצע: `polls - avg`, ואחר כך `.std()`. התוצאה היא Series בשם `sd`, עם ערך אחד לכל מפלגה.

**ואז רצפה:** מפלגה שכל הסקרים נותנים לה בדיוק 8 תקבל סטיית תקן 0. כלומר "אין שום אי-ודאות", וזה לא נכון. הגבל מלמטה ל-**1 מנדט**: `.clip(lower=1.0)`.

**רמז:** pandas מחסר Series מכל שורה של DataFrame לפי שמות העמודות, ולכן `polls - avg` עובד ישירות.

In [2]:
# TODO
sd = (polls - avg).std().clip(lower=1.0)

In [3]:
# בדיקה
assert list(sd.index) == list(avg.index)
assert (sd >= 1.0).all()
pd.DataFrame({"average": avg, "sd": sd.round(2)}).sort_values("sd", ascending=False)

,average,sd
id,,
likud,21.7,3.83
amcha-israel,2.3,2.14
zionist-home,3.3,1.89
yachad,11.1,1.83
yashar,21.7,1.69
yisrael-beitenu,7.8,1.44
shas,7.7,1.02
democrats,9.0,1.00
joint-list,8.1,1.00


**לפני שממשיכים — הסתכל על הטבלה:** למי סטיית התקן הכי גדולה? האם זה אומר שהמפלגה הזו באמת "לא יציבה"? רמז: הממוצע באתר מתקן את **הנטייה של כל בית סקרים** (house effects), אבל סטיית התקן כאן מחושבת מהסקרים הגולמיים. אם ערוץ אחד נותן לליכוד בקביעות 5 מנדטים יותר, מה זה עושה ל-`sd`?

## משימה 2 — בחירות מדומות אחת

`simulate_once(rng)` מחזירה מערך numpy של 14 מספרים, מנדטים לכל מפלגה, בסדר של `avg.index`:

1. מגרילים לכל מפלגה מנדטים מהתפלגות נורמלית: `rng.normal(avg.values, sd.values)`.
2. חותכים מספרים שליליים ל-0: `np.clip(x, 0, None)`.
3. **אחוז חסימה:** מפלגה מתחת ל-`THRESHOLD_SEATS` מקבלת 0.
4. **מנרמלים ל-120:** מכפילים ב-`TOTAL / x.sum()`. המנדטים של מפלגות שנפלו מתחלקים בין שאר המפלגות, באופן יחסי.

**רמז לשלב 3:** אינדקס בוליאני, `x[x < THRESHOLD_SEATS] = 0`.

In [4]:
# TODO
def simulate_once(rng):
    x = rng.normal(avg.values, sd.values)
    x.clip(x,0,None)
    x[x < THRESHOLD_SEATS] = 0
    x = x * (TOTAL / x.sum())
    return x

In [5]:
# בדיקה
x = simulate_once(np.random.default_rng(0))
assert x.shape == (len(avg),)
assert abs(x.sum() - TOTAL) < 1e-9
assert ((x == 0) | (x >= THRESHOLD_SEATS)).all(), "אף מפלגה לא אמורה לקבל מנדטים מתחת לאחוז החסימה"
assert not np.array_equal(x, simulate_once(np.random.default_rng(1))), "seed אחר צריך לתת בחירות אחרות"
pd.Series(x, index=avg.index).round(1).sort_values(ascending=False)

id
yashar               24.1
likud                21.7
democrats             9.8
joint-list            9.7
yachad                9.7
otzma                 9.1
yahadut-hatorah       8.2
religious-zionism     7.9
shas                  7.7
raam                  7.2
yisrael-beitenu       4.9
amcha-israel          0.0
kachol-lavan          0.0
zionist-home          0.0
dtype: float64

## משימה 3 — עשרת אלפים בחירות

`simulate(n, seed)` מחזירה DataFrame: `n` שורות, אחת לכל בחירות מדומות, ועמודה לכל מפלגה.

אפשר לקרוא ל-`simulate_once` בלולאה. **אתגר רשות:** לעשות את זה בלי לולאה, עם `rng.normal(..., size=(n, len(avg)))`. כל השלבים של משימה 2 עובדים גם על מטריצה, ואז הריצה מהירה פי מאות. בנרמול צריך `x.sum(axis=1, keepdims=True)`.

In [6]:
# TODO
def simulate(n, seed=0):
    np.random.default_rng(seed)
    rng = np.random.default_rng(seed)
    x = rng.normal(avg.values, sd.values, size=(n, len(avg)))
    
    x = np.clip(x, 0, None)
    
    x[x < THRESHOLD_SEATS] = 0
    
    x = x * (TOTAL / x.sum(axis=1, keepdims=True))
    
    return pd.DataFrame(x, columns=avg.index)


In [7]:
# בדיקה
S = simulate(10_000)
assert S.shape == (10_000, len(avg)) and list(S.columns) == list(avg.index)
assert np.allclose(S.sum(axis=1), TOTAL)
assert simulate(50, seed=3).equals(simulate(50, seed=3)), "אותו seed צריך לתת אותה תוצאה"
S.describe().T[["mean", "std", "min", "max"]].round(1)

,mean,std,min,max
id,,,,
amcha-israel,1.1,2.2,0.0,9.9
democrats,9.2,1.1,5.7,14.1
joint-list,8.3,1.1,4.2,12.9
kachol-lavan,0.0,0.0,0.0,0.0
likud,22.2,3.4,9.5,34.6
otzma,8.1,1.1,4.6,12.3
raam,5.0,1.8,0.0,9.0
religious-zionism,6.3,1.2,0.0,10.6
shas,7.9,1.1,0.0,12.8


## משימה 4 — מה הסיכוי של קואליציה?

`coalition_probability(S, members)` מקבלת את הסימולציה ורשימת מזהי מפלגות, ומחזירה את **החלק מהבחירות המדומות שבהן סכום המנדטים שלהן הוא לפחות 61**.

שורה אחת: לבחור את העמודות, לסכום לכל שורה, להשוות ל-`MAJORITY` ולחשב ממוצע. ממוצע של True/False הוא בדיוק החלק שבו התנאי מתקיים.

In [8]:
# TODO
def coalition_probability(S, members):
    return (S[members].sum(axis = 1) >= MAJORITY).mean()

In [9]:
# בדיקה
assert coalition_probability(S, list(avg.index)) == 1.0, "כל הכנסת תמיד 120"
assert coalition_probability(S, ["kachol-lavan"]) == 0.0
right = ["likud", "otzma", "religious-zionism", "shas", "yahadut-hatorah"]
p = coalition_probability(S, right)
assert 0 <= p <= 1
print(f"ליכוד + עוצמה + הציונות הדתית + ש\"ס + יהדות התורה: {p:.1%}")

ליכוד + עוצמה + הציונות הדתית + ש"ס + יהדות התורה: 0.7%


## משימה 5 — לשחק עם זה

1. **סיכוי לעבור את אחוז החסימה:** לכל מפלגה, `(S > 0).mean()`. מי על הסף?
2. **נסה כמה קואליציות.** מה קורה לסיכוי של קואליציית הימין אם מוסיפים לה את ישראל ביתנו? ואם מוסיפים את עמך ישראל, שעל סף אחוז החסימה?
3. **כמה זה מדויק?** עם `n` בחירות מדומות, טעות התקן של סיכוי `p` היא `√(p(1−p)/n)`. כמה היא ב-`n = 10,000`, כש-`p = 0.5`? האם צריך יותר?

In [10]:
# TODO
# 1. סיכוי לעבור את אחוז החסימה
passes = (S > 0).mean().sort_values()
print("סיכוי לעבור את אחוז החסימה:")
print(passes.round(3))

# 2. כמה מוסיפה כל מפלגה לקואליציית הימין והחרדים
print()
for extra in [[], ["yisrael-beitenu"], ["amcha-israel"], ["yisrael-beitenu", "amcha-israel"]]:
    p = coalition_probability(S, right + extra)
    print(f"ימין + חרדים {'+ ' + ' + '.join(extra) if extra else ''}: {p:.1%}")

# 3. כמה מדויקת הסימולציה
n = len(S)
se = (0.5 * 0.5 / n) ** 0.5
print(f"\nטעות התקן במקרה הגרוע (p=0.5, n={n:,}): ±{se:.2%}")

סיכוי לעבור את אחוז החסימה:
id
kachol-lavan         0.000
amcha-israel         0.222
zionist-home         0.382
raam                 0.906
religious-zionism    0.988
yisrael-beitenu      0.996
shas                 1.000
democrats            1.000
joint-list           1.000
likud                1.000
otzma                1.000
yachad               1.000
yahadut-hatorah      1.000
yashar               1.000
dtype: float64

ימין + חרדים : 0.7%
ימין + חרדים + yisrael-beitenu: 47.2%
ימין + חרדים + amcha-israel: 1.9%
ימין + חרדים + yisrael-beitenu + amcha-israel: 59.8%

טעות התקן במקרה הגרוע (p=0.5, n=10,000): ±0.50%


## שאלות לחשיבה — לפני שזה נכנס לאתר

1. **אי-תלות:** המודל מגריל כל מפלגה בנפרד. אבל טעויות של סקרים נוטות להיות **מתואמות**: אם הסקרים מפספסים את הימין, הם בדרך כלל מפספסים כמה מפלגות ימין ביחד. האם אי-תלות מגדילה או מקטינה את הסיכוי של **גוש** להגיע ל-61? רמז: מה קורה לשונות של סכום, כשמשתנים חיוביים מתואמים?
2. **house effects:** ‏`sd` מחושב מהסקרים הגולמיים. האם זה מגזים באי-הוודאות, או ממעיט בה?
3. **רצפה של 1:** למה 1 ולא 0.5 או 2? מה זה משנה למפלגות שעל סף אחוז החסימה?
4. **14 יום:** האם חלון ארוך יותר נותן הערכה טובה יותר של `sd`, או גרועה יותר? מה קורה כשיש מגמה אמיתית בתוך החלון?

את ההחלטות על 1 ו-2 נצטרך לקבל לפני שמעבירים לאתר. תכתוב לי מה חשבת.